### SFT training for the qwen 1.5B

In [ ]:
# Install specialized libraries for RTX 4500 Ada performance
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install flash-attn --no-build-isolation

In [ ]:
import json
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# 1. Configuration
model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit"
max_seq_length = 2048 # Pushing context length for complex SRE states
dtype = None # None for auto-detection (will use bf16 on RTX 4500 Ada)
load_in_4bit = True # Keeps memory low to allow massive batch sizes

# 2. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Higher rank for better intelligence
    target_modules = ["q_proj", "k_v_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. Prepare Dataset
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Format matching your generate_dataset logic
        text = f"### System:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n{output}"
        texts.append(text)
    return { "text" : texts }

with open("sft_dataset.json", "r") as f:
    raw_data = json.load(f)

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. Training Arguments Optimized for 48GB VRAM
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 8, # Pushed for 48GB GPU
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 500 samples / (8*4) ≈ 15 steps per epoch. 60 steps ≈ 4 epochs.
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), # Native on RTX 4500
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_sft",
    ),
)

# 5. Execute Training
trainer_stats = trainer.train()

# 6. Save for RL Phase
model.save_pretrained_merged("sre_model_sft_final", tokenizer, save_method = "merged_16bit")
print("SFT Training Complete. Model saved for RL Phase.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

# Create plots folder if it doesn't exist (failsafe)
os.makedirs("plots", exist_ok=True)

# Extract history from trainer
history = trainer.state.log_history
df = pd.DataFrame(history)

# Filter out empty rows (logging steps vs eval steps)
df_logs = df[df['loss'].notna()]

plt.figure(figsize=(12, 5))

# Plot 1: SFT Loss Curve
plt.subplot(1, 2, 1)
plt.plot(df_logs['step'], df_logs['loss'], color='#6366f1', linewidth=2)
plt.title("SFT Training Loss\n(Format & Reasoning Adherence)")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)

# Plot 2: Learning Rate Schedule
plt.subplot(1, 2, 2)
plt.plot(df_logs['step'], df_logs['learning_rate'], color='#10b981', linewidth=2)
plt.title("SFT Learning Rate\n(Warmup & Decay)")
plt.xlabel("Step")
plt.ylabel("LR")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("plots/sft_training_metrics.png", dpi=300)
plt.show()

print("SFT plots saved as plots/sft_training_metrics.png")